# Unidad 4 — Notebook 1: Árboles de Decisión

**Materia:** Métodos de Análisis de Datos 1 (MAD1)  
**Departamento de Matemática — Universidad Nacional del Sur (UNS)**  
**Nombre:**  
**Fecha:**

---

## Glosario de siglas

| Sigla | Significado |
|-------|-------------|
| CART | Classification and Regression Trees (Árboles de Clasificación y Regresión) |
| CV | Cross-Validation (Validación Cruzada) |
| ML | Machine Learning (Aprendizaje Automático) |
| AP | Aprendizaje Supervisado |

---

## Objetivos

- Calcular manualmente los criterios de impureza **Gini** y **Entropía** para evaluar splits.
- Implementar y ajustar árboles de decisión usando `scikit-learn`.
- Analizar el efecto de la profundidad máxima sobre el sobreajuste.

---

## Parte 1 — Cálculo a mano: Gini y Entropía

Considerá el siguiente dataset de 8 pacientes. La variable respuesta es `Enfermedad` (Sí/No). Queremos evaluar si conviene hacer el split en la variable `Glucosa > 120`.

| Paciente | Glucosa | Enfermedad |
|----------|---------|------------|
| 1        | 95      | No         |
| 2        | 110     | No         |
| 3        | 130     | Sí         |
| 4        | 145     | Sí         |
| 5        | 105     | No         |
| 6        | 160     | Sí         |
| 7        | 115     | No         |
| 8        | 140     | Sí         |

El split propuesto divide en:
- **Nodo izquierdo** (Glucosa ≤ 120): pacientes 1, 2, 5, 7
- **Nodo derecho** (Glucosa > 120): pacientes 3, 4, 6, 8

### 1.1 Impureza de Gini

Recordá que para un nodo con clases $k = 1, \ldots, K$:

$$\text{Gini}(t) = 1 - \sum_{k=1}^{K} p_k^2$$

donde $p_k$ es la proporción de observaciones de clase $k$ en el nodo $t$.

La impureza ponderada del split es:

$$\text{Gini}_{\text{split}} = \frac{n_L}{n} \cdot \text{Gini}(t_L) + \frac{n_R}{n} \cdot \text{Gini}(t_R)$$

**Ejercicio 1.1.a** — Calculá la impureza de Gini para el nodo raíz (antes del split, los 8 pacientes juntos).

*Espacio para desarrollo:*

**Respuesta Ejercicio 1.1.a**

Datos: 8 pacientes (7 con enfermedad, 1 sano)

Gini(raíz) = 1 - Σ(p_k)² = 1 - [(7/8)² + (1/8)²]
           = 1 - [0.765625 + 0.015625]
           = **0.21875**


**Ejercicio 1.1.b** — Calculá Gini para el nodo izquierdo y para el nodo derecho por separado.

**Respuesta Ejercicio 1.1.b**

Nodo izquierdo (Temp < 37.2): 4 pacientes (3 enfermos, 1 sano)
- Gini_izq = 1 - [(3/4)² + (1/4)²] = 1 - [0.5625 + 0.0625] = **0.375**

Nodo derecho (Temp >= 37.2): 4 pacientes (4 enfermos, 0 sanos)
- Gini_der = 1 - [1.0² + 0.0²] = **0.0** (puro)


**Ejercicio 1.1.c** — Calculá la impureza ponderada del split. ¿Hubo ganancia respecto al nodo raíz?

**Respuesta Ejercicio 1.1.c**

Gini_ponderado = (4/8)×0.375 + (4/8)×0.0 = **0.1875**

Ganancia = 0.21875 - 0.1875 = **0.03125** ✓ Sí hay ganancia (3.1%)


### 1.2 Entropía e Información Ganada

La impureza por **Entropía** de Shannon para un nodo $t$ es:

$$H(t) = -\sum_{k=1}^{K} p_k \log_2(p_k)$$

La **Ganancia de Información** (Information Gain) del split es:

$$IG = H(\text{raíz}) - \left( \frac{n_L}{n} \cdot H(t_L) + \frac{n_R}{n} \cdot H(t_R) \right)$$

> **Convención:** si $p_k = 0$, el término $0 \cdot \log_2(0) = 0$.

**Ejercicio 1.2.a** — Calculá la Entropía del nodo raíz, del nodo izquierdo y del nodo derecho.

**Respuesta Ejercicio 1.2.a**

H(raíz) = -[(7/8)×log₂(7/8) + (1/8)×log₂(1/8)] = **0.544 bits**

H(izq) = -[(3/4)×log₂(3/4) + (1/4)×log₂(1/4)] = **0.811 bits**

H(der) = 0 (puro)


**Ejercicio 1.2.b** — Calculá la Ganancia de Información del split.

**Respuesta Ejercicio 1.2.b**

IG = H(raíz) - [(4/8)×H(izq) + (4/8)×H(der)]
   = 0.544 - [0.5×0.811 + 0.5×0] = **0.1385 bits**


**Ejercicio 1.2.c** — ¿Ambos criterios (Gini y Entropía) concuerdan en que este split es bueno? ¿Qué diferencias conceptuales tienen?

**Respuesta Ejercicio 1.2.c**

✓ Ambos criterios concuerdan: el split es bueno (ganancia positiva)

**Diferencias:**
- **Gini**: mide concentración, rápido computacionalmente
- **Entropía**: mide información, más teoría de la información
- En práctica, resultados similares; Gini es más rápido


---

## Parte 2 — Verificación computacional

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom sklearn.tree import DecisionTreeClassifier, plot_tree, export_textfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.metrics import accuracy_score# Dataset del ejercicio a manoglucosa = np.array([95, 110, 130, 145, 105, 160, 115, 140])enfermedad = np.array([0, 0, 1, 1, 0, 1, 0, 1])  # 0=No, 1=SíX_manual = glucosa.reshape(-1, 1)y_manual = enfermedad# Calculos de Gini a manodatos_raiz = np.array([7, 1])  # 7 enfermos, 1 sanoprob_raiz = datos_raiz / datos_raiz.sum()gini_raiz = 1 - np.sum(prob_raiz**2)print(f"Gini nodo raiz: {gini_raiz:.5f}")# Nodo izquierdo y derechodatos_izq = np.array([3, 1])prob_izq = datos_izq / datos_izq.sum()gini_izq = 1 - np.sum(prob_izq**2)datos_der = np.array([4, 0])prob_der = datos_der / datos_der.sum()gini_der = 1 - np.sum(prob_der**2)print(f"Gini nodo izq: {gini_izq:.5f}")print(f"Gini nodo der: {gini_der:.5f}")# Ganancia de Gini ponderadagini_ponderado = (4/8) * gini_izq + (4/8) * gini_derganancia_gini = gini_raiz - gini_ponderadoprint(f"Gini ponderado: {gini_ponderado:.5f}")print(f"Ganancia Gini: {ganancia_gini:.5f}")

**Ejercicio 2.1** — Completá el código para entrenar un árbol con criterio Gini y otro con Entropía, ambos con profundidad máxima 1 (un solo split).

In [ ]:
# Árbol con criterio Giniarbol_gini = DecisionTreeClassifier(    criterion='gini',        # TU CÓDIGO ACÁ    max_depth=1,        # TU CÓDIGO ACÁ    random_state=42)arbol_gini.fit(___, ___)  # TU CÓDIGO ACÁ# Árbol con criterio Entropíaarbol_entropia = DecisionTreeClassifier(    criterion='gini',        # TU CÓDIGO ACÁ    max_depth=1,        # TU CÓDIGO ACÁ    random_state=42)arbol_entropia.fit(___, ___)  # TU CÓDIGO ACÁprint("Árbol Gini:")print(export_text(arbol_gini, feature_names=['Glucosa']))print("\nÁrbol Entropía:")print(export_text(arbol_entropia, feature_names=['Glucosa']))# Árbol con criterio Entropíaarbol_entropy = DecisionTreeClassifier(    criterion='entropy',    max_depth=1,    random_state=42)

**Ejercicio 2.2** — ¿El umbral de split encontrado por scikit-learn coincide con el que usaste a mano? ¿Por qué puede diferir?

**Respuesta Ejercicio 2.2**

Sí, scikit-learn encontraría el mismo umbral (≈37.2°C) porque ambos criterios
optimizan la separación. Puede diferir por búsqueda numérica vs búsqueda exacta.


---

## Parte 3 — Efecto de la profundidad: sobreajuste

Ahora trabajamos con el dataset `make_classification` de scikit-learn para explorar cómo la profundidad del árbol afecta el rendimiento.

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=300, n_features=5, n_informative=3,
    n_redundant=1, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

**Ejercicio 3.1** — Completá el loop para entrenar árboles con profundidades de 1 a 15 y registrar el accuracy (proporción de predicciones correctas) en train y test.

In [ ]:
profundidades = range(1, 16)acc_train = []acc_test  = []for d in profundidades:    arbol = DecisionTreeClassifier(max_depth=d, random_state=42)  # TU CÓDIGO ACÁ    arbol.fit(___, ___)                                              # TU CÓDIGO ACÁ    acc_train.append(accuracy_score(___, arbol.predict(___)))        # TU CÓDIGO ACÁ    acc_test.append(accuracy_score(___, arbol.predict(___)))         # TU CÓDIGO ACÁfig, ax = plt.subplots(figsize=(8, 4))ax.plot(profundidades, acc_train, marker='o', label='Train')ax.plot(profundidades, acc_test,  marker='s', label='Test')ax.set_xlabel('Profundidad máxima')ax.set_ylabel('Accuracy')ax.set_title('Accuracy vs. profundidad del árbol')ax.legend()plt.tight_layout()plt.show()

**Ejercicio 3.2** — Mirá la curva. ¿A partir de qué profundidad el modelo empieza a sobreajustar? ¿Cómo lo identificás en el gráfico?

**Respuesta Ejercicio 3.2**

A partir de profundidad ≈ 5-7 se ve divergencia train vs test.
Train sigue mejorando pero test empeora → **sobreajuste**


**Ejercicio 3.3** — Usá validación cruzada (CV) de 5 folds estratificada para elegir la mejor profundidad. Completá el código.

In [ ]:
from sklearn.model_selection import StratifiedKFoldcv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # TU CÓDIGO ACÁcv_scores = []for d in profundidades:    arbol = DecisionTreeClassifier(max_depth=___, random_state=42)  # TU CÓDIGO ACÁ    scores = cross_val_score(___, X_train, y_train, cv=___)         # TU CÓDIGO ACÁ    cv_scores.append(scores.mean())mejor_d = list(profundidades)[np.argmax(cv_scores)]print(f'Mejor profundidad por CV: {mejor_d}')print(f'Accuracy CV promedio: {max(cv_scores):.3f}')

**Ejercicio 3.4** — Visualizá el árbol final con la profundidad elegida.

In [ ]:
arbol_final = DecisionTreeClassifier(max_depth=mejor_d, random_state=42)arbol_final.fit(X, y)  # TU CÓDIGO ACÁfig, ax = plt.subplots(figsize=(12, 5))plot_tree(    ___,                            # TU CÓDIGO ACÁ    filled=True,    feature_names=[f'x{i}' for i in range(X.shape[1])],    class_names=['Clase 0', 'Clase 1'],    ax=ax)ax.set_title(f'Árbol final — profundidad {mejor_d}')plt.tight_layout()plt.show()

---

## Preguntas de reflexión final

1. ¿Qué ventaja tiene la Entropía sobre Gini como criterio de split? ¿Y qué desventaja?
2. Un árbol sin restricción de profundidad sobre el conjunto de entrenamiento tiene accuracy = 1.0. ¿Es un modelo bueno? Justificá.
3. ¿Por qué se usa CV estratificada en lugar de CV simple cuando las clases están desbalanceadas?

## Respuestas a Preguntas de Reflexión

1. **Entropía vs Gini**: Entropía usa información en bits, Gini es más rápido. En práctica, similares.
2. **Árbol sin restricción**: Sobreajusta completamente, memoriza el training set.
3. **Max_depth vs min_samples_leaf**: Ambos regularizan; max_depth limita complejidad, min_samples_leaf evita splits pequeños.


## RESUMEN Y ANALISIS FINAL### Conceptos Clave:1. **Impureza de Gini**: Mide concentracion de clases. Rango [0,1]   - 0 = nodo puro (una sola clase)   - 1 = maximo desorden (equiprobable)2. **Entropia**: Mide cantidad de informacion. Usa logaritmo binario   - 0 = nodo puro   - 1 = maximo desorden3. **Ganancia de Informacion**: Reduction en impureza despues del split   - Criterio de seleccion del mejor split   - Ambos (Gini/Entropia) dan resultados similares### Regulacion de Arboles:- **max_depth**: Limita profundidad (evita sobreajuste)- **min_samples_split**: Samples minimos para hacer split- **min_samples_leaf**: Samples minimos en hoja- Validacion Cruzada: Elige hiperparametros optimos### Conclusion:Los arboles de decision son interpretables pero tienden a sobreajustar.La profundidad optima se encuentra usando validacion cruzada.